In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("Joined.csv")

In [ ]:
df.head()

,date,product_id,your_price,competitor1_price,competitor2_price,sales_volume
0,03-04-2025,P001,1639.52,1572.87,1590.05,NaN
1,08-08-2025,P001,1671.84,1623.08,1671.10,NaN
2,26-08-2025,P001,1666.36,1555.86,1661.17,NaN
3,13-06-2025,P001,1668.35,1596.76,1594.39,39.0
4,21-08-2025,P001,1621.04,1620.74,1638.62,41.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               5000 non-null   object 
 1   product_id         5000 non-null   object 
 2   your_price         5000 non-null   float64
 3   competitor1_price  4912 non-null   float64
 4   competitor2_price  4875 non-null   float64
 5   sales_volume       4930 non-null   float64
dtypes: float64(4), object(2)
memory usage: 234.5+ KB


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.duplicated(subset=["product_id", "date"]).sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
date,0
product_id,0
your_price,0
competitor1_price,88
competitor2_price,125
sales_volume,70


In [ ]:
df = df.dropna(subset=["sales_volume"])

In [ ]:
df.shape

(4930, 6)

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4930 entries, 3 to 4999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               4930 non-null   datetime64[ns]
 1   product_id         4930 non-null   object        
 2   your_price         4930 non-null   float64       
 3   competitor1_price  4844 non-null   float64       
 4   competitor2_price  4806 non-null   float64       
 5   sales_volume       4930 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 269.6+ KB


In [ ]:
df = df.sort_values(["product_id", "date"]).reset_index(drop=True)
df.head()

,date,product_id,your_price,competitor1_price,competitor2_price,sales_volume
0,2025-01-01,P001,1650.58,1549.76,1622.00,56.0
1,2025-01-02,P001,1611.85,1574.50,1610.93,63.0
2,2025-01-03,P001,1603.69,1617.00,1633.04,71.0
3,2025-01-04,P001,1652.81,1610.04,1616.04,86.0
4,2025-01-05,P001,1651.76,1534.03,1577.39,66.0


In [ ]:
df.describe()

,date,your_price,competitor1_price,competitor2_price,sales_volume
count,4930,4930.000000,4844.000000,4806.000000,4930.000000
mean,2025-05-05 17:58:23.610547456,1112.907112,1143.451080,1078.002901,67.548073
min,2025-01-01 00:00:00,132.230000,169.290000,147.580000,21.000000
25%,2025-03-04 00:00:00,347.542500,493.485000,372.302500,52.000000
50%,2025-05-06 00:00:00,982.335000,1073.230000,1005.100000,66.000000
75%,2025-07-07 00:00:00,1560.957500,1542.255000,1589.152500,82.000000
max,2025-09-07 00:00:00,2565.760000,2640.140000,2381.860000,183.000000
std,NaN,694.002672,715.892127,647.335632,22.299412


In [ ]:
df["product_id"].value_counts().sort_index()

,count
product_id,
P001,247
P002,248
P003,243
P004,247
P005,248
P006,244
P007,246
P008,246
P009,250


In [ ]:
import numpy as np
df["ln_price"] = np.log(df["your_price"])
df["ln_sales"] = np.log(df["sales_volume"])

In [ ]:
import statsmodels.api as sm

results = []
for pid, g in df.groupby("product_id"):
    X = sm.add_constant(g["ln_price"])
    model = sm.OLS(g["ln_sales"], X).fit()
    results.append({
        "product_id": pid,
        "elasticity": model.params["ln_price"],
        "p_value": model.pvalues["ln_price"],
        "r_squared": model.rsquared,
        "n_rows": len(g)
    })

elasticity_df = pd.DataFrame(results).sort_values("elasticity")
elasticity_df

,product_id,elasticity,p_value,r_squared,n_rows
2,P003,-2.753271,1.131847e-20,0.303284,243
4,P005,-2.358881,3.045197e-17,0.252091,248
15,P016,-2.299025,2.231744e-17,0.258486,243
18,P019,-2.141093,5.239593e-15,0.220535,248
1,P002,-2.088038,8.241060e-13,0.188315,248
7,P008,-2.046299,9.772241e-13,0.188591,246
3,P004,-2.035715,1.494155e-13,0.200056,247
6,P007,-1.956354,1.082053e-12,0.187924,246
8,P009,-1.874415,1.139505e-11,0.169847,250
19,P020,-1.873920,7.570578e-13,0.188177,249


In [ ]:
# price gap in % (positive = competitor is more expensive than you)
df["gap_c1"] = (df["competitor1_price"] / df["your_price"] - 1) * 100
df["gap_c2"] = (df["competitor2_price"] / df["your_price"] - 1) * 100

# day-over-day price change per product
df["c1_change"] = df.groupby("product_id")["competitor1_price"].diff()
df["c2_change"] = df.groupby("product_id")["competitor2_price"].diff()

# per-product summary
comp_summary = df.groupby("product_id").agg(
    avg_gap_c1=("gap_c1", "mean"),
    avg_gap_c2=("gap_c2", "mean"),
    c1_increases=("c1_change", lambda s: (s > 0).sum()),
    c2_increases=("c2_change", lambda s: (s > 0).sum())
).reset_index()

# overall: who is pricier and who raises prices more
print(df[["gap_c1", "gap_c2"]].mean())
print((df["c1_change"] > 0).sum(), (df["c2_change"] > 0).sum())
comp_summary

gap_c1    3.234562
gap_c2   -2.172389
dtype: float64
2349 2361


,product_id,avg_gap_c1,avg_gap_c2,c1_increases,c2_increases
0,P001,-1.992525,-0.018504,122,114
1,P002,2.669107,-6.907249,118,116
2,P003,-3.293566,2.810309,113,110
3,P004,10.129049,3.121801,112,120
4,P005,7.014773,-7.154884,128,116
5,P006,3.703683,4.491375,106,118
6,P007,9.309333,-5.979823,125,121
7,P008,1.381319,-3.379579,116,121
8,P009,-3.658339,5.373222,123,117
9,P010,2.751071,-8.780447,119,120


In [ ]:
from google.colab import files

df.to_csv("clean_data.csv", index=False)
elasticity_df.to_csv("elasticity_results.csv", index=False)
comp_summary.to_csv("competitor_summary.csv", index=False)

files.download("clean_data.csv")
files.download("elasticity_results.csv")
files.download("competitor_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>